In [1]:
import os
import random
import shutil
from pathlib import Path
from PIL import Image
from tqdm import tqdm




SEED = 42
random.seed(SEED)

TARGET_SIZE = 299
MAX_RESIZE = 320

OUTPUT_ROOT = Path("merged_dataset")

TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2




STYLEGAN_REAL = Path("140k-real-and-fake-faces/real_vs_fake/train/real")
STYLEGAN_FAKE = Path("140k-real-and-fake-faces/real_vs_fake/train/fake")

REALVSFAKE_REAL = Path("realvsfake-81k-by-wish/RealVsFake/Real")
REALVSFAKE_FAKE = Path("realvsfake-81k-by-wish/RealVsFake/Fake")

SYNTHETICEYE = {
    "min-dalle": Path("syntheticeye-diffusion-faces/SyntheticEye Diffusion Faces/min-dalle"),
    "openjourney": Path("syntheticeye-diffusion-faces/SyntheticEye Diffusion Faces/openjourney"),
    "stable-diffusion": Path("syntheticeye-diffusion-faces/SyntheticEye Diffusion Faces/stable-diffusion"),
}

STABLE_DIFF_FACE = Path("stable-diffusion-face-dataset")




def preprocess_and_save(src, dst):
    try:
        img = Image.open(src).convert("RGB")

        w, h = img.size
        scale = MAX_RESIZE / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.BICUBIC)

        w, h = img.size
        left = (w - TARGET_SIZE) // 2
        top = (h - TARGET_SIZE) // 2
        img = img.crop((left, top, left + TARGET_SIZE, top + TARGET_SIZE))

        img.save(dst, format="PNG")
    except Exception as e:
        print(f"Skipping {src}: {e}")

def split_list(items):
    random.shuffle(items)
    n = len(items)
    t = int(n * TRAIN_RATIO)
    v = int(n * VAL_RATIO)
    return items[:t], items[t:t+v], items[t+v:]

def collect_images(folder):
    return [p for p in folder.rglob("*") if p.suffix.lower() in [".jpg", ".png", ".jpeg"]]




for split in ["train", "val", "test_id", "test_crossgen"]:
    for cls in ["real", "fake_gan", "fake_diffusion"]:
        (OUTPUT_ROOT / split / cls).mkdir(parents=True, exist_ok=True)




real_images = collect_images(STYLEGAN_REAL) + collect_images(REALVSFAKE_REAL)
train, val, test = split_list(real_images)

for split, data in zip(["train", "val", "test_id"], [train, val, test]):
    for img in tqdm(data, desc=f"Real {split}"):
        preprocess_and_save(img, OUTPUT_ROOT / split / "real" / img.name)




gan_images = collect_images(STYLEGAN_FAKE)
train, val, test = split_list(gan_images)

for split, data in zip(["train", "val", "test_id"], [train, val, test]):
    for img in tqdm(data, desc=f"GAN {split}"):
        preprocess_and_save(img, OUTPUT_ROOT / split / "fake_gan" / img.name)




diff_train, diff_val, diff_test = [], [], []


rvf_diff = collect_images(REALVSFAKE_FAKE)
t, v, te = split_list(rvf_diff)
diff_train += t
diff_val += v
diff_test += te


for name, path in SYNTHETICEYE.items():
    imgs = collect_images(path)
    t, v, _ = split_list(imgs)
    diff_train += t
    diff_val += v


sd_crossgen = collect_images(STABLE_DIFF_FACE)
for img in sd_crossgen:
    preprocess_and_save(img, OUTPUT_ROOT / "test_crossgen" / "fake_diffusion" / img.name)


for split, data in zip(
    ["train", "val", "test_id"],
    [diff_train, diff_val, diff_test]
):
    for img in tqdm(data, desc=f"Diffusion {split}"):
        preprocess_and_save(img, OUTPUT_ROOT / split / "fake_diffusion" / img.name)

print("✅ Dataset merging & preprocessing complete.")


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:

PATH_140K = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"


PATH_CIFAKE = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images"


MERGED_BASE = "/kaggle/working/fft_merged_balanced_50"


In [4]:
for split in ["train", "val", "test"]:
    for cls in ["REAL", "FAKE"]:
        os.makedirs(os.path.join(MERGED_BASE, split, cls), exist_ok=True)

print("✅ Created merged dataset folders")


✅ Created merged dataset folders


In [5]:
def copy_images(src_dir, dst_dir, limit, prefix):
    files = [f for f in os.listdir(src_dir) if f.lower().endswith((".jpg",".png",".jpeg"))]
    random.shuffle(files)
    files = files[:limit]

    for f in tqdm(files, desc=f"Copying → {os.path.basename(dst_dir)}", leave=False):
        shutil.copy(
            os.path.join(src_dir, f),
            os.path.join(dst_dir, prefix + f)
        )


In [6]:

gan_train_real = os.path.join(PATH_140K, "train", "real")
gan_train_fake = os.path.join(PATH_140K, "train", "fake")


gan_val_real = os.path.join(PATH_140K, "valid", "real")
gan_val_fake = os.path.join(PATH_140K, "valid", "fake")


gan_test_real = os.path.join(PATH_140K, "test", "real")
gan_test_fake = os.path.join(PATH_140K, "test", "fake")


In [7]:
cifake_train_fake = os.path.join(PATH_CIFAKE, "train", "FAKE")
cifake_test_fake  = os.path.join(PATH_CIFAKE, "test",  "FAKE")


In [8]:
dst_train_real = os.path.join(MERGED_BASE, "train", "REAL")
dst_train_fake = os.path.join(MERGED_BASE, "train", "FAKE")

dst_val_real   = os.path.join(MERGED_BASE, "val", "REAL")
dst_val_fake   = os.path.join(MERGED_BASE, "val", "FAKE")

dst_test_real  = os.path.join(MERGED_BASE, "test", "REAL")
dst_test_fake  = os.path.join(MERGED_BASE, "test", "FAKE")


In [9]:

copy_images(gan_train_real, dst_train_real, 50000, "real_")


copy_images(gan_val_real, dst_val_real, 10000, "real_")


copy_images(gan_test_real, dst_test_real, 10000, "real_")

print("✅ REAL images copied")


✅ REAL images copied


In [10]:
copy_images(gan_train_fake, dst_train_fake, 25000, "gan_")
copy_images(cifake_train_fake, dst_train_fake, 25000, "diff_")


In [11]:
copy_images(gan_val_fake, dst_val_fake, 5000, "gan_")
copy_images(cifake_test_fake, dst_val_fake, 5000, "diff_")


In [13]:
copy_images(gan_test_fake, dst_test_fake, 5000, "gan_")
copy_images(cifake_test_fake, dst_test_fake, 5000, "diff_")

print("✅ FAKE images copied (50% GAN / 50% Diffusion)")


✅ FAKE images copied (50% GAN / 50% Diffusion)


In [17]:
def count_dir(path):
    return len([f for f in os.listdir(path) if f.lower().endswith((".jpg",".png",".jpeg"))])

print("\n📊 FINAL DATASET COUNTS")
print("TRAIN REAL:", count_dir(dst_train_real))
print("TRAIN FAKE:", count_dir(dst_train_fake))

print("VAL REAL:", count_dir(dst_val_real))
print("VAL FAKE:", count_dir(dst_val_fake))

print("TEST REAL:", count_dir(dst_test_real))
print("TEST FAKE:", count_dir(dst_test_fake))



📊 FINAL DATASET COUNTS
TRAIN REAL: 50000
TRAIN FAKE: 50000
VAL REAL: 10000
VAL FAKE: 10000
TEST REAL: 10000
TEST FAKE: 10000


In [18]:
def fft_log_magnitude(gray_img_np):

    f = np.fft.fft2(gray_img_np)
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)
    logmag = np.log1p(mag)
    return logmag

def high_pass_mask(shape, radius_ratio=0.15):

    H, W = shape
    cy, cx = H // 2, W // 2
    r = int(min(H, W) * radius_ratio)
    Y, X = np.ogrid[:H, :W]
    dist = (Y - cy)**2 + (X - cx)**2
    mask = (dist >= r*r).astype(np.float32)
    return mask

def apply_hpf(logmag, radius_ratio=0.15):
    mask = high_pass_mask(logmag.shape, radius_ratio)
    return logmag * mask


In [19]:
def sample_paths(root_dir, n=2000):
    all_files = []
    for cls in ["REAL","FAKE"]:
        cls_dir = os.path.join(root_dir, cls)
        files = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir) if f.lower().endswith((".jpg",".jpeg",".png"))]
        all_files.extend(files)
    random.shuffle(all_files)
    return all_files[:n]

def compute_fft_mean_std(train_root, img_size=224, radius_ratio=0.15, n=2000):
    paths = sample_paths(train_root, n=n)
    vals = []

    for p in tqdm(paths, desc="Computing FFT mean/std"):
        img = Image.open(p).convert("L").resize((img_size, img_size))
        arr = np.array(img).astype(np.float32)
        logmag = fft_log_magnitude(arr)
        logmag = apply_hpf(logmag, radius_ratio=radius_ratio)
        vals.append(logmag)

    vals = np.stack(vals, axis=0)  
    mean = vals.mean()
    std = vals.std() + 1e-8
    return float(mean), float(std)

TRAIN_ROOT = os.path.join(MERGED_BASE, "train")
fft_mean, fft_std = compute_fft_mean_std(TRAIN_ROOT, n=1500)
print("FFT Mean:", fft_mean, "FFT Std:", fft_std)


Computing FFT mean/std: 100%|██████████| 1500/1500 [00:04<00:00, 353.21it/s]


FFT Mean: 5.89819860458374 FFT Std: 2.155052661895752


In [20]:
class FFTDataset(Dataset):
    def __init__(self, root_dir, img_size=224, radius_ratio=0.15, mean=0.0, std=1.0):
        self.root_dir = root_dir
        self.img_size = img_size
        self.radius_ratio = radius_ratio
        self.mean = mean
        self.std = std

        self.samples = []
        for label, cls in enumerate(["FAKE", "REAL"]):  
            cls_dir = os.path.join(root_dir, cls)
            files = [f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg",".jpeg",".png"))]
            for f in files:
                self.samples.append((os.path.join(cls_dir, f), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("L").resize((self.img_size, self.img_size))
        arr = np.array(img).astype(np.float32)

        logmag = fft_log_magnitude(arr)
        logmag = apply_hpf(logmag, radius_ratio=self.radius_ratio)


        logmag = (logmag - self.mean) / self.std


        x = torch.tensor(logmag, dtype=torch.float32).unsqueeze(0)
        y = torch.tensor(label, dtype=torch.long)
        return x, y


In [21]:
BATCH_SIZE = 64
NUM_WORKERS = 2  

fft_train_ds = FFTDataset(os.path.join(MERGED_BASE, "train"), mean=fft_mean, std=fft_std)
fft_val_ds   = FFTDataset(os.path.join(MERGED_BASE, "val"),   mean=fft_mean, std=fft_std)
fft_test_ds  = FFTDataset(os.path.join(MERGED_BASE, "test"),  mean=fft_mean, std=fft_std)

fft_train_loader = DataLoader(fft_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
fft_val_loader   = DataLoader(fft_val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
fft_test_loader  = DataLoader(fft_test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("Train size:", len(fft_train_ds), "Val:", len(fft_val_ds), "Test:", len(fft_test_ds))


Train size: 100000 Val: 20000 Test: 20000


In [22]:
fft_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

old_conv = fft_model.conv1
new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
fft_model.conv1 = new_conv

fft_model.fc = nn.Linear(fft_model.fc.in_features, 2)
fft_model = fft_model.to(device)

print("FFT model ready.")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 180MB/s] 


FFT model ready.


In [23]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(loader, desc="Train", leave=False):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(loader, desc="Val/Test", leave=False):
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)

        total_loss += loss.item() * x.size(0)
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total


In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(fft_model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS = 5
best_val_acc = 0.0
SAVE_PATH = "/kaggle/working/best_fft_cifake_hpf.pth"

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(fft_model, fft_train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(fft_model, fft_val_loader, criterion)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"FFT Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"FFT Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(fft_model.state_dict(), SAVE_PATH)
        print("✅ Saved best model:", SAVE_PATH)



Epoch 1/5
FFT Train Loss: 0.4002 | Train Acc: 0.7995
FFT Val   Loss: 0.3549 | Val   Acc: 0.8305
✅ Saved best model: /kaggle/working/best_fft_cifake_hpf.pth



Epoch 2/5
FFT Train Loss: 0.3375 | Train Acc: 0.8417
FFT Val   Loss: 0.3483 | Val   Acc: 0.8401
✅ Saved best model: /kaggle/working/best_fft_cifake_hpf.pth



Epoch 3/5
FFT Train Loss: 0.2926 | Train Acc: 0.8678
FFT Val   Loss: 0.3614 | Val   Acc: 0.8372



Epoch 4/5
FFT Train Loss: 0.2262 | Train Acc: 0.9041
FFT Val   Loss: 0.3930 | Val   Acc: 0.8417
✅ Saved best model: /kaggle/working/best_fft_cifake_hpf.pth



Epoch 5/5
FFT Train Loss: 0.1368 | Train Acc: 0.9449
FFT Val   Loss: 0.5978 | Val   Acc: 0.8238


In [25]:

fft_model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
fft_model.eval()

@torch.no_grad()
def evaluate_metrics(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    for x, y in tqdm(loader, desc="Metrics", leave=False):
        x = x.to(device)
        out = model(x)
        pred = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(pred.tolist())
        all_labels.extend(y.numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', pos_label=0)  


    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=["Fake","Real"])

    return acc, prec, rec, f1, cm, report

test_acc, test_prec, test_rec, test_f1, cm, report = evaluate_metrics(fft_model, fft_test_loader)

print(f"\nFFT Test Accuracy : {test_acc:.4f}")
print(f"FFT Precision     : {test_prec:.4f}")
print(f"FFT Recall        : {test_rec:.4f}")
print(f"FFT F1-score      : {test_f1:.4f}")
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)



FFT Test Accuracy : 0.8431
FFT Precision     : 0.8927
FFT Recall        : 0.7801
FFT F1-score      : 0.8326

Confusion Matrix:
 [[7801 2199]
 [ 938 9062]]

Classification Report:
               precision    recall  f1-score   support

        Fake       0.89      0.78      0.83     10000
        Real       0.80      0.91      0.85     10000

    accuracy                           0.84     20000
   macro avg       0.85      0.84      0.84     20000
weighted avg       0.85      0.84      0.84     20000



In [26]:
import torch.nn.functional as F

def predict_fft_image(model, img_path, device, img_size=224, radius_ratio=0.15, mean=0.0, std=1.0):
    img = Image.open(img_path).convert("L").resize((img_size, img_size))
    arr = np.array(img).astype(np.float32)

    logmag = fft_log_magnitude(arr)
    logmag = apply_hpf(logmag, radius_ratio=radius_ratio)
    logmag = (logmag - mean) / std

    x = torch.tensor(logmag, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)  

    model.eval()
    with torch.no_grad():
        out = model(x)
        prob = F.softmax(out, dim=1).cpu().numpy()[0]  
        pred = int(np.argmax(prob))

    label_map = {0: "FAKE", 1: "REAL"}

    return {
        "prediction": label_map[pred],
        "fake_confidence": float(prob[0]),
        "real_confidence": float(prob[1])
    }


In [27]:
img_path = "/kaggle/input/fft-test-images/gemini-image-1.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.2239508032798767
Real confidence: 0.7760492563247681


In [28]:
img_path = "/kaggle/input/fft-test-images/gemini-image-2.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.324656218290329
Real confidence: 0.6753438115119934


In [29]:
img_path = "/kaggle/input/fft-test-images/gemini-image-3.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.08702889084815979
Real confidence: 0.9129711389541626


In [30]:
img_path = "/kaggle/input/fft-test-images/gemini-image-4.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: FAKE
Fake confidence: 0.87027907371521
Real confidence: 0.12972094118595123


In [31]:
img_path = "/kaggle/input/fft-test-images/gemini-image-5.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.23398274183273315
Real confidence: 0.7660172581672668


In [34]:
img_path = "/kaggle/input/fft-test-images/gemini-image-6.jpeg"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.023828057572245598
Real confidence: 0.9761719703674316


In [36]:
img_path = "/kaggle/input/fft-test-images/gemini-image-7.jpeg"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.008768798783421516
Real confidence: 0.9912311434745789


In [37]:
img_path = "/kaggle/input/fft-test-images/gemini-image-8.jpeg"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.324375182390213
Real confidence: 0.6756248474121094


In [38]:
img_path = "/kaggle/input/fft-test-images/gpt-image-1.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: FAKE
Fake confidence: 0.8956416845321655
Real confidence: 0.1043582633137703


In [39]:
img_path = "/kaggle/input/fft-test-images/gpt-image-2.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.007968618534505367
Real confidence: 0.9920313358306885


In [40]:
img_path = "/kaggle/input/fft-test-images/gpt-image-3.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.4297473430633545
Real confidence: 0.5702526569366455


In [41]:
img_path = "/kaggle/input/fft-test-images/gpt-image-4.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.10723946243524551
Real confidence: 0.8927605152130127


In [42]:
img_path = "/kaggle/input/fft-test-images/gpt-image-5.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.212641179561615
Real confidence: 0.7873588800430298


In [43]:
img_path = "/kaggle/input/fft-test-images/gpt-image-6.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.1662740707397461
Real confidence: 0.8337258696556091


In [44]:
img_path = "/kaggle/input/fft-test-images/gpt-image-7.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.17398406565189362
Real confidence: 0.8260159492492676
